# ChatPromptTemplate

> 基本概念: 多种调用方式,消息占位符

## 1. 基础用法：from_template

In [1]:
import os
import dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 使用 from_template 创建提示词模板
prompt = ChatPromptTemplate.from_template("请用一句话简单介绍{topic}是什么？")

# 查看生成的消息结构
rprint("模板消息：")
rprint(prompt.messages)

# 格式化模板
formatted = prompt.format_messages(topic="Docker")
rprint("\n格式化后：")
for msg in formatted:
    rprint(f"  {msg.type}: {msg.content}")

# 结合 LLM 使用
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)
chain = prompt | llm
response = chain.invoke({"topic": "Docker"})
rprint(f"\n回答：{response}")

模板消息：

[
    HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=['topic'],
            input_types={},
            partial_variables={},
            template='请用一句话简单介绍{topic}是什么？'
        ),
        additional_kwargs={}
    )
]

格式化后：

human: 请用一句话简单介绍Docker是什么？

回答：content='Docker 
是一个开源的应用容器化平台，它能将应用程序及其依赖环境打包成一个轻量级、可移植的"容器"，实现在任何环境中都能一致地
运行。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 57, 
'prompt_tokens': 260, 'total_tokens': 317, 'completion_tokens_details': {'accepted_prediction_tokens': None, 
'audio_tokens': None, 'reasoning_tokens': 14, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': 
{'audio_tokens': None, 'cached_tokens': 192}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 
'system_fingerprint': None, 'id': '918bf6e95793431a963d257ce1e25ef2', 'finish_reason': 'stop', 'logprobs': None} 
id='lc_run--019f1c59-c246-7200-ab66-8e20002b8487-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={'input_tokens': 260, 'output_tokens': 57, 'total_tokens': 317, 'input_token_details': 
{'cache_read': 192}, 'output_token_details': {'reasoning': 14}}

## 2. from_messages：指定消息角色

In [2]:
import os
import dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# from_messages：可以指定 SystemMessage 和 HumanMessage
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}助手，回答要简洁，不超过{count}个字。"),
    ("human", "{question}"),
])

# 查看消息结构
print("模板消息：")
for msg in prompt.messages:
    rprint(f"  {msg.__class__.__name__}: {msg.prompt.template}")

# 格式化
formatted = prompt.format_messages(role="Python编程", count=500, question="什么是装饰器？")
print("\n格式化后：")
for msg in formatted:
    print(f"  {msg.type}: {msg.content}")

# 结合 LLM 使用
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)
chain = prompt | llm
response = chain.invoke({"role": "Python编程", "count": 50, "question": "什么是装饰器？"})
rprint(f"\n回答：{response.content}")

模板消息：


SystemMessagePromptTemplate: 你是一个{role}助手，回答要简洁，不超过{count}个字。

HumanMessagePromptTemplate: {question}


格式化后：
  system: 你是一个Python编程助手，回答要简洁，不超过500个字。
  human: 什么是装饰器？


回答：装饰器是一个函数，它接受另一个函数作为参数，并返回一个新函数，通常用于在不修改原函数代码的情况下扩展其功能。
比如添加日志或权限验证。简单来说，装饰器就像是给函数“穿一件外套”。

## 3. partial：预填充部分变量

In [7]:
from langchain_core.prompts import ChatPromptTemplate

# 创建模板，有多个变量
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}助手，回答要简洁。"),
    ("human", "{question}"),
])

# partial：预先填充 role 变量，后续只需传 question
python_prompt = prompt.partial(role="Python编程",  question="Python中如何读取JSON文件？")
java_prompt = prompt.partial(role="Java编程")

# 查看 partial 后的消息
print("Python 模板：")
print(f"  {python_prompt.messages}")

print("Java 模板：")
print(f"  {java_prompt.messages[0].prompt.template}")

# 格式化时只需传入剩余变量
formatted = python_prompt.format_messages(question="什么是GIL？")
print(f"\n格式化：{formatted[1].content}")

Python 模板：
  [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='你是一个{role}助手，回答要简洁。'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]
Java 模板：
  你是一个{role}助手，回答要简洁。

格式化：什么是GIL？


## 4. 多轮对话模板（含历史消息）

In [ ]:
import os
import dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# 多轮对话模板：使用 MessagesPlaceholder 插入历史消息
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 模拟历史对话
history = [
    HumanMessage(content="你好，我叫小明"),
    AIMessage(content="你好小明！有什么可以帮你的吗？"),
]

# 格式化：传入历史消息和新问题
formatted = prompt.format_messages(
    history=history,
    input="你还记得我叫什么吗？"
)

print("格式化后的历史消息：")
for msg in formatted:
    print(f"  {msg.type}: {msg.content}")

# 结合 LLM 使用
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)
chain = prompt | llm
response = chain.invoke({"history": history, "input": "你还记得我叫什么吗？"})
print(f"\n回答：{response.content}")

## 5. with_output_parser：结构化输出

In [ ]:
import os
import dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# 使用 LCEL 管道符组合：prompt | llm | output_parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}助手，回答要简洁。"),
    ("human", "{question}"),
])

# output_parser 将 AIMessage 转为纯字符串
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

chain = prompt | llm | StrOutputParser()

# 直接得到字符串，而非 AIMessage 对象
response = chain.invoke({"role": "Python编程", "question": "什么是装饰器？"})
print(f"类型：{type(response)}")
print(f"内容：{response}")